# Experiment 005: base-to-GRPO 0.5B, exploratory extension (Colab)

GRPO straight on Qwen2.5-0.5B with no sft step, seed 0, frozen protocol v1. Stages are
base -> grpo -> eval, so there is no sft training or sft evaluation in this run.

**Scientific status:** The original study plan specified the base-to-GRPO comparison at 3B.
This run extends that comparison to 0.5B as an exploratory experiment under protocol v1. Experiment 002 is useful
as protocol-development context, but it ran on a T4 in FP16 and helped select the final settings.
It should not be used as the clean final sft-then-grpo counterpart to 005. That comparison needs
a frozen 0.5B sft-then-grpo rerun on A100 with the same seed.

**Experiment number:** earlier planning reserved 005 for 3B seed 1. This notebook follows the
new decision to use 005 for the 0.5B base-to-GRPO extension.

**GPU: use A100.** This starts the standardised A100 plan for the final scale comparisons.
`dtype: auto` selects BF16 on A100, matching experiments 003 and 004 and the planned reruns.
Do not reuse experiment 002 base files because they were produced on T4 in FP16. 005 must
calculate its own base evaluation on A100. One session should be enough, but resume is still
available if the runtime disconnects.

On a disconnect, re-run the full-run cell. Finished stages skip and grpo resumes from the latest
checkpoint saved on drive.

## 1. Check the GPU

This run is fixed to A100 so the cell stops before any work if a different GPU is attached.

In [ ]:
!nvidia-smi -L
import torch

print('cuda:', torch.cuda.is_available(), '| torch:', torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('A GPU runtime is required for this experiment.')

gpu_name = torch.cuda.get_device_name(0)
print('gpu:', gpu_name)
if 'A100' not in gpu_name:
    raise RuntimeError('Experiment 005 is fixed to A100. Change the runtime GPU before continuing.')
if not torch.cuda.is_bf16_supported():
    raise RuntimeError('BF16 support was not detected on this runtime.')
print('A100 confirmed, dtype auto will use BF16')

GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-60ec52bb-2122-9234-6729-969a0952a7c8)
cuda: True | torch: 2.11.0+cu128
gpu: NVIDIA A100-SXM4-40GB
A100 confirmed, dtype auto will use BF16


## 2. Clone the repo (automatic)

This cell clones the main branch of the public repository without asking for a GitHub token. For online experiment tracking, add `WANDB_API_KEY` under the key icon in Colab Secrets and grant this notebook access. If using a private fork, set `REPO_SLUG` to its owner and name, then add a read-only `GH_TOKEN` for that fork. A missing token for a private fork triggers a one-time prompt.


In [ ]:
import base64
import getpass
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def clone_repo():
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, 'clone', '--branch', 'main', '--single-branch',
                    '--depth', '1', repo_url, str(repo_root)], env=env, check=True)


if not repo_root.exists():
    clone_repo()
if not (repo_root / '.git').exists():
    raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
origin = subprocess.check_output(
    ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
).strip()
if origin != repo_url:
    raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')

# Use the scripts from this checkout even if another experiment is open.
os.chdir(repo_root / '001_Code')
print('working in:', os.getcwd())


GitHub token source: Colab Secrets
Cloning into 'INM363-IndividualProject-Dissertation'...
remote: Enumerating objects: 763, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 763 (delta 3), reused 17 (delta 3), pack-reused 743 (from 1)
Receiving objects: 100% (763/763), 10.91 MiB | 18.46 MiB/s, done.
Resolving deltas: 100% (422/422), done.
working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies

Keep the runtime's own cuda torch and pin the libraries used by the experiment.

In [ ]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml
# remove torchao because quantization is not used here
!pip uninstall -q -y torchao
print('deps installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 170.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 55.5 MB/s eta 0:00:00
deps installed


## 4. W&B key (automatic)

Reads `WANDB_API_KEY` from Colab Secrets and writes it to the file expected by the run code.
If the secret is unavailable, it falls back to a hidden one-time prompt.

In [ ]:
import os

# use the Colab secret first and prompt for it only when unavailable
def _clean_wandb_key(value):
    if isinstance(value, dict):
        value = value.get('value')
    if not isinstance(value, str):
        return None
    return value.strip() or None

wkey_source = 'Colab Secrets'
try:
    from google.colab import userdata
    wkey = _clean_wandb_key(userdata.get('WANDB_API_KEY'))
except Exception:
    wkey = None
if not wkey:
    import getpass
    wkey_source = 'hidden prompt'
    wkey = _clean_wandb_key(getpass.getpass('W&B API key: '))
if not wkey:
    raise RuntimeError('WANDB_API_KEY was not supplied through Colab Secrets or the hidden prompt.')
with open('wandb_api_key.txt', 'w') as key_file:
    key_file.write(wkey)
wkey = None
print('W&B key source:', wkey_source)
print('have key file:', os.path.exists('wandb_api_key.txt'))

W&B key source: Colab Secrets
have key file: True


## 5. Mount Google Drive

Results and checkpoints are written directly to drive, so a disconnected session can resume.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/inm363', exist_ok=True)
print('drive mounted -> /content/drive/MyDrive/inm363')

Mounted at /content/drive
drive mounted -> /content/drive/MyDrive/inm363


## 6. Keep the base evaluation on A100

Do not copy the experiment 002 base cache. It was produced on T4 in FP16, while this run uses
A100 in BF16. The first 005 session calculates a fresh base evaluation and later sessions reuse
only that 005 cache.

This cell marks the 005 cache for A100. It stops if an older unmarked cache already exists, which
protects against accidentally keeping files copied by the previous notebook version.

In [ ]:
from pathlib import Path

base_cache = Path('/content/drive/MyDrive/inm363/results/base_evals/005_COLAB_FULL_B05')
gpu_marker = base_cache / '.gpu_plan'
existing_files = list(base_cache.iterdir()) if base_cache.is_dir() else []
if existing_files and not gpu_marker.exists():
    raise RuntimeError(
        'An unmarked 005 base cache already exists. Check it, then delete or rename that folder before this A100 run: '
        + str(base_cache)
    )

# mark this base cache as A100 before the run starts
base_cache.mkdir(parents=True, exist_ok=True)
gpu_marker.write_text('A100 BF16\n')
print('005 base cache plan: A100 BF16')
print('a fresh base eval will run now' if not existing_files else 'the marked 005 base cache can resume')

005 base cache plan: A100 BF16
a fresh base eval will run now


## 7. Build the data splits

Materialises gsm8k for sft compatibility, mmlu dev and validation for grpo, and mmlu test for
evaluation. The build also checks for leakage.

In [ ]:
!python scripts/build_data.py

README.md: 100% 7.93k/7.93k [00:00<00:00, 25.0MB/s]
main/train-00000-of-00001.parquet: 100% 2.31M/2.31M [00:01<00:00, 1.75MB/s]
main/test-00000-of-00001.parquet: 100% 419k/419k [00:01<00:00, 342kB/s]
Generating train split: 100% 7473/7473 [00:00<00:00, 403673.47 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 266000.91 examples/s]
README.md: 100% 53.2k/53.2k [00:00<00:00, 85.8MB/s]
dataset_infos.json: 100% 138k/138k [00:00<00:00, 186MB/s]
high_school_psychology/test-00000-of-000(…): 100% 92.8k/92.8k [00:01<00:00, 74.5kB/s]
high_school_psychology/validation-00000-(…): 100% 15.2k/15.2k [00:01<00:00, 12.4kB/s]
high_school_psychology/dev-00000-of-0000(…): 100% 5.18k/5.18k [00:01<00:00, 4.09kB/s]
Generating test split: 100% 545/545 [00:00<00:00, 149727.89 examples/s]
Generating validation split: 100% 60/60 [00:00<00:00, 28626.80 examples/s]
Generating dev split: 100% 5/5 [00:00<00:00, 2656.98 examples/s]
marketing/test-00000-of-00001.parquet: 100% 37.3k/37.3k [00:01<00:00, 2

## 8. Wiring check

Quick check that 005 loads as `B05`, `BASE_GRPO`, seed 0 and plans only base eval, grpo and grpo eval.

In [ ]:
!python scripts/run.py --config configs/005_colab_full_b05_base_grpo_seed0.yaml

env=COLAB mode=FULL scale=B05 condition=BASE_GRPO
model=Qwen/Qwen2.5-0.5B device=cuda stages=['base_eval', 'grpo', 'grpo_eval']
steps=500 N=8 seed=0
trackers=['tensorboard', 'wandb']
results -> /content/drive/MyDrive/inm363/results/005_COLAB_FULL_B05_base_grpo_seed0/  checkpoints -> /content/drive/MyDrive/inm363/checkpoints/005_COLAB_FULL_B05_base_grpo_seed0/
local smoke wiring ok


## 9. Full run

Base -> grpo -> eval, with metrics, per-example predictions, checkpoints, manifest and wandb.
The first A100 session calculates the 005 base evaluation. Later sessions skip its completed
files and continue from the saved stage.

If the runtime disconnects, re-run this cell. Finished stages skip and an interrupted grpo stage
resumes from its latest checkpoint on drive.

Watch `frac_reward_zero_std`, reward accuracy, KL, clipping and projected time. The same frozen
abort rules used for 002, 003 and 004 apply here.

In [ ]:
# use expandable segments to reduce cuda memory problems
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/run.py --config configs/005_colab_full_b05_base_grpo_seed0.yaml --stage all

Streaming output truncated to the last 5000 lines.
│ │               │ is actually   │               │              │           │ │
│ │               │ correct.      │               │              │           │ │
│ │               │ let me        │               │              │           │ │
│ │               │ know,the      │               │              │           │ │
│ │               │ default data  │               │              │           │ │
│ │               │ desgin should │               │              │           │ │
│ │               │ be            │               │              │           │ │
│ │               │ auto-create-… │               │              │           │ │
│ │               │ In Apache     │               │              │           │ │
│ │               │ Spark, you    │               │              │           │ │
│ │               │ can create a  │               │              │           │ │
│ │               │ dataframe     │               │       

## 10. Peek at the results

The final run folder is `results/005_COLAB_FULL_B05_base_grpo_seed0`. The shared base files stay
under `results/base_evals/005_COLAB_FULL_B05`.

In [ ]:
import json, os
from config import load_config, run_tag
from scripts.run import base_eval_dir

cfg = load_config('configs/005_colab_full_b05_base_grpo_seed0.yaml')
tag = run_tag(cfg)
base = os.path.join(cfg.output_root, 'results', tag)
bdir = base_eval_dir(cfg)
print('results dir:', base)
print('files:', os.listdir(base) if os.path.isdir(base) else 'missing')
print('shared base evals:', os.listdir(bdir) if os.path.isdir(bdir) else 'missing')

# print only files that already exist
for name in ('summary', 'manifest'):
    path = os.path.join(base, f'{name}.json')
    if os.path.exists(path):
        print(f'\n{name}:')
        with open(path) as result_file:
            print(json.dumps(json.load(result_file), indent=2)[:2500])
    else:
        print(f'{name}.json not there yet (run not finished)')

results dir: /content/drive/MyDrive/inm363/results/005_COLAB_FULL_B05_base_grpo_seed0
files: ['run.log', 'grpo_trainlog.json', 'grpo_preds.jsonl', 'grpo_eval.json', 'grpo_logprob_preds.jsonl', 'grpo_logprob.json', 'summary.json', 'manifest.json']
shared base evals: ['.gpu_plan', 'base_preds.jsonl', 'base_eval.json', 'base_logprob_preds.jsonl', 'base_logprob.json']

summary:
{
  "scale": "B05",
  "mode": "FULL",
  "condition": "BASE_GRPO",
  "seed": 0,
  "base_acc": {
    "easy": 0.20819112627986347,
    "medium": 0.18719211822660098,
    "hard": 0.13131313131313133
  },
  "sft_acc": null,
  "grpo_acc": {
    "easy": 0.48236632536973834,
    "medium": 0.3727422003284072,
    "hard": 0.255050505050505
  },
  "reference": "base",
  "grpo_delta": {
    "easy": 0.27417519908987487,
    "medium": 0.18555008210180624,
    "hard": 0.1237373737373737
  },
  "base_logprob_acc": {
    "easy": 0.658703071672355,
    "medium": 0.5582922824302134,
    "hard": 0.37626262626262624
  },
  "sft_logprob_